# ViT Baseline -- Phase 1: MAE Pretraining on Unlabelled Jet Images
**Standard Vision Transformer baseline for comparison against XCiT.**

Structurally identical to `xcit-jet-images-mae.ipynb` but uses **quadratic self-attention**
$\mathcal{O}(N^2)$ instead of XCiT cross-covariance attention $\mathcal{O}(Nd^2/h)$.

| Component | ViT (this notebook) | XCiT |
|---|---|---|
| Attention | Standard MHSA $\mathcal{O}(N^2 d)$ | Cross-covariance $\mathcal{O}(Nd^2/h)$ |
| Spatial comm. | Implicit via attention | Explicit LPI (depth-wise 3x3 convs) |
| Pos. encoding | Learned 1D | Sinusoidal 2D |
| Params | ~6M | ~7M |
| MAE mask ratio | 60% | 60% |
| Loss | Activity-weighted MSE patch-normalised | Same |

**Output:** `best_vit_mae.pth` loaded by `vit-multitask.ipynb`.

## 1 . Installs

In [5]:
# !pip install torch torchvision einops h5py scikit-learn matplotlib


## 2 . Imports & device

In [6]:
import numpy as np, math, os, time
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps')
print(f'Device: {DEVICE}')


Device: mps


## 3 . Hyperparameters
Architecture constants **must match `vit-multitask.ipynb`** exactly.
Training hyperparameters are identical to the XCiT MAE notebook for a fair comparison.

In [ ]:
# -- Data -------------------------------------------------------------------
H5_PATH     = '/kaggle/input/jet-unlabelled/Dataset_Specific_Unlabelled.h5'
H5_PATH     = './Dataset_Specific_Unlabelled.h5'
BATCH_SIZE  = 16
NUM_WORKERS = 0

# -- Training (identical to XCiT MAE) ----------------------------------------
EPOCHS        = 1
LR            = 1e-3
WEIGHT_DECAY  = 0.05
WARMUP_EPOCHS = 0
MASK_RATIO    = 0.60    # same sparse-image adaptation as XCiT MAE

# -- ViT encoder (FIXED - must match vit-multitask.ipynb) --------------------
IMAGE_SIZE   = 125
PATCH_SIZE   = 5
IN_CHANNELS  = 8
DIM          = 192      # same as XCiT for fair comparison
DEPTH        = 8
NUM_HEADS    = 8        # head_dim = 192/8 = 24
MLP_RATIO    = 4
DROPOUT      = 0.0      # disabled during MAE pretraining
DROP_PATH    = 0.0      # disabled during MAE pretraining
NUM_PATCHES  = (IMAGE_SIZE // PATCH_SIZE) ** 2   # 625

# -- MAE decoder (discarded after pretraining) --------------------------------
DECODER_DIM   = 96
DECODER_DEPTH = 4
DECODER_HEADS = 3

print(f'Patches      : {NUM_PATCHES}')
print(f'Masked       : {int(NUM_PATCHES*MASK_RATIO)}/{NUM_PATCHES}  ({MASK_RATIO:.0%})')
print(f'Encoder      : dim={DIM}  depth={DEPTH}  heads={NUM_HEADS}')
print(f'Attention    : Standard MHSA  O(N^2 * d)')
print(f'Decoder      : dim={DECODER_DIM}  depth={DECODER_DEPTH}  (discarded)')


Patches      : 625
Masked       : 375/625  (60%)
Encoder      : dim=192  depth=8  heads=8
Attention    : Standard MHSA  O(N^2 * d)
Decoder      : dim=96  depth=4  (discarded)


## 4 . Dataset
Identical to the XCiT MAE notebook. Streams H5 directly, applies `log1p` + per-sample max-norm.

In [ ]:
class UnlabelledJetDataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        with h5py.File(h5_path, 'r') as f:
            self.N = f['jet'].shape[0]
        print(f'Dataset: {self.N} unlabelled images')
    def __len__(self): return self.N
    def __getitem__(self, idx):
        with h5py.File(self.h5_path, 'r') as f:
            x = f['jet'][idx][:30000].astype(np.float32)
        x = np.log1p(x)
        x = x / (x.max() + 1e-8)
        return torch.from_numpy(x).permute(2, 0, 1)  # (8, 125, 125)

dataset    = UnlabelledJetDataset(H5_PATH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True)
print(f'Batches/epoch: {len(dataloader)}')


Dataset: 60000 unlabelled images
Batches/epoch: 3750


## 5 . ViT Building Blocks

### Standard Multi-Head Self-Attention (MHSA)
$$\\text{Attention}(Q,K,V) = \\text{Softmax}\\!\\left(QK^\\top / \\sqrt{d_k}\\right)V$$

The attention map is $(N \\times N)$ -- quadratic in token count.
XCiT's attention map is $(d/h \\times d/h)$ -- linear in $N$.

At 625 patches:
- ViT MHSA: $625^2 \\times 192 \\approx 75\\text{M}$ ops per layer
- XCiT XCA: $625 \\times 24^2 \\approx 360\\text{K}$ ops per layer

In [9]:
class DropPath(nn.Module):
    def __init__(self, p=0.0):
        super().__init__(); self.p = p
    def forward(self, x):
        if not self.training or self.p == 0.0: return x
        keep = 1 - self.p
        mask = (torch.rand((x.shape[0],)+(1,)*(x.ndim-1), device=x.device)<keep).float()
        return x * mask / keep


class MHSA(nn.Module):
    # Standard Multi-Head Self-Attention. Complexity O(N^2 * d).
    def __init__(self, dim, num_heads=8, qkv_bias=True, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.qkv       = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj      = nn.Sequential(nn.Linear(dim, dim), nn.Dropout(dropout))

    def forward(self, x):
        B, N, C = x.shape
        h   = self.num_heads
        qkv = self.qkv(x).reshape(B, N, 3, h, C//h).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]             # (B, h, N, d/h)
        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, h, N, N)
        attn = attn.softmax(dim=-1)
        out  = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(out)


class ViTBlock(nn.Module):
    # Standard ViT block: Pre-LN MHSA + Pre-LN FFN.
    def __init__(self, dim, num_heads, mlp_ratio=4, dropout=0.0, drop_path=0.0):
        super().__init__()
        self.norm1     = nn.LayerNorm(dim)
        self.norm2     = nn.LayerNorm(dim)
        self.attn      = MHSA(dim, num_heads=num_heads, dropout=dropout)
        self.ffn       = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * mlp_ratio, dim), nn.Dropout(dropout),
        )
        self.drop_path = DropPath(drop_path) if drop_path > 0 else nn.Identity()

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.ffn(self.norm2(x)))
        return x


print('ViT building blocks defined.')


ViT building blocks defined.


## 6 . ViT Encoder
Uses a **learned 1D positional embedding** (standard ViT) rather than sinusoidal 2D (XCiT).
During MAE pretraining only visible patch tokens are processed.
The CLS token is present for Phase 2 weight compatibility but unused during pretraining.

In [10]:
class ViTEncoder(nn.Module):
    # Standard ViT encoder for MAE pretraining.
    # Encodes VISIBLE patches only. CLS token present for Phase 2 compatibility.
    def __init__(self,
                 image_size  = IMAGE_SIZE,
                 patch_size  = PATCH_SIZE,
                 in_channels = IN_CHANNELS,
                 dim         = DIM,
                 depth       = DEPTH,
                 num_heads   = NUM_HEADS,
                 mlp_ratio   = MLP_RATIO,
                 dropout     = DROPOUT,
                 drop_path   = DROP_PATH):
        super().__init__()
        self.patch_size = patch_size
        self.H = self.W = image_size // patch_size
        num_patches     = self.H * self.W

        # Patch embedding (same conv-based design as XCiT)
        self.patch_embed = nn.Conv2d(in_channels, dim,
                                     kernel_size=patch_size, stride=patch_size)
        self.emb_norm    = nn.LayerNorm(dim)

        # Learned 1D positional embedding (key difference from XCiT sincos 2D)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # CLS token (used in Phase 2, not during MAE pretraining)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        # ViT transformer blocks
        dpr = [x.item() for x in torch.linspace(0, drop_path, depth)]
        self.blocks = nn.ModuleList([
            ViTBlock(dim, num_heads, mlp_ratio, dropout, dpr[i])
            for i in range(depth)
        ])
        self.norm = nn.LayerNorm(dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x, visible_mask):
        # x            : (B, C, H_img, W_img)
        # visible_mask : (B, N) bool - True = visible
        # Returns: visible_tokens (B, N_vis, dim), H, W
        H, W = self.H, self.W
        B    = x.shape[0]

        # Patch embed + positional encoding on ALL patches
        x_emb = self.patch_embed(x).flatten(2).transpose(1, 2).contiguous()
        x_emb = self.emb_norm(x_emb + self.pos_embed)   # (B, N, dim)

        # Select visible patches only
        N_vis = visible_mask[0].sum().item()
        x_vis = x_emb[visible_mask].reshape(B, N_vis, -1)

        # Run ViT blocks
        for blk in self.blocks:
            x_vis = blk(x_vis)
        x_vis = self.norm(x_vis)
        return x_vis, H, W


enc = ViTEncoder().to(DEVICE)
n   = sum(p.numel() for p in enc.parameters() if p.requires_grad)
print(f'ViT encoder parameters: {n:,}')
del enc


ViT encoder parameters: 3,718,464


## 7 . MAE Decoder
**Identical** to the XCiT MAE decoder so any downstream difference is due to the encoder only.

In [11]:
class MAEDecoder(nn.Module):
    # Lightweight MAE decoder. Identical to XCiT MAE decoder.
    def __init__(self,
                 encoder_dim   = DIM,
                 decoder_dim   = DECODER_DIM,
                 decoder_depth = DECODER_DEPTH,
                 decoder_heads = DECODER_HEADS,
                 patch_size    = PATCH_SIZE,
                 in_channels   = IN_CHANNELS,
                 num_patches   = NUM_PATCHES):
        super().__init__()
        patch_pixels     = in_channels * patch_size * patch_size  # 200
        self.enc_to_dec  = nn.Linear(encoder_dim, decoder_dim, bias=True)
        self.mask_token  = nn.Parameter(torch.zeros(1, 1, decoder_dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        self.dec_pos_emb = nn.Embedding(num_patches, decoder_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=decoder_dim, nhead=decoder_heads,
            dim_feedforward=decoder_dim*4,
            dropout=0.0, activation='gelu',
            batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=decoder_depth)
        self.norm        = nn.LayerNorm(decoder_dim)
        self.pred        = nn.Linear(decoder_dim, patch_pixels)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, visible_tokens, visible_mask, num_patches):
        B      = visible_tokens.shape[0]
        device = visible_tokens.device
        vis    = self.enc_to_dec(visible_tokens)
        tokens = self.mask_token.expand(B, num_patches, -1).clone()
        tokens[visible_mask] = vis.reshape(-1, vis.shape[-1])
        pos_ids = torch.arange(num_patches, device=device).unsqueeze(0).expand(B, -1)
        tokens  = tokens + self.dec_pos_emb(pos_ids)
        tokens  = self.norm(self.transformer(tokens))
        return self.pred(tokens)


## 8 . Full ViT-MAE Model
Masking strategy and loss **identical** to `xcit-jet-images-mae.ipynb`:
- Random 60% masking
- Patch-normalised reconstruction targets
- Activity-weighted loss (10x on active patches)

In [12]:
class ViTMAE(nn.Module):
    # Masked Autoencoder with standard ViT encoder.
    # Loss and masking identical to XCiTMAE for fair comparison.
    ACTIVE_PATCH_WEIGHT = 10.0

    def __init__(self, mask_ratio=MASK_RATIO):
        super().__init__()
        self.mask_ratio  = mask_ratio
        self.num_patches = NUM_PATCHES
        self.patch_size  = PATCH_SIZE
        self.in_channels = IN_CHANNELS
        self.encoder     = ViTEncoder()
        self.decoder     = MAEDecoder()

    def random_mask(self, B, N, device):
        N_vis = int(N * (1 - self.mask_ratio))
        ids   = torch.rand(B, N, device=device).argsort(dim=1)
        vis   = torch.zeros(B, N, dtype=torch.bool, device=device)
        vis.scatter_(1, ids[:, :N_vis], True)
        return vis, ~vis

    def patchify(self, imgs):
        ps = self.patch_size
        B, C, H, W = imgs.shape
        H_p, W_p   = H // ps, W // ps
        x = imgs.reshape(B, C, H_p, ps, W_p, ps)
        x = x.permute(0, 2, 4, 1, 3, 5).contiguous()
        return x.reshape(B, H_p * W_p, C * ps * ps)

    def patch_normalise(self, patches):
        mean = patches.mean(dim=-1, keepdim=True)
        std  = patches.std(dim=-1, keepdim=True).clamp(min=1e-6)
        return (patches - mean) / std, mean, std

    def activity_weights(self, patches):
        activity  = patches.abs().mean(dim=-1)
        threshold = activity.mean() * 0.1
        return torch.where(activity > threshold,
                           torch.full_like(activity, self.ACTIVE_PATCH_WEIGHT),
                           torch.ones_like(activity))

    def forward(self, imgs):
        B, C, H, W        = imgs.shape
        N                  = self.num_patches
        vis_mask, m_mask   = self.random_mask(B, N, imgs.device)
        vis_tokens, Hg, Wg = self.encoder(imgs, vis_mask)
        pred               = self.decoder(vis_tokens, vis_mask, N)
        target_raw         = self.patchify(imgs)
        target_norm, _, _  = self.patch_normalise(target_raw)
        weights            = self.activity_weights(target_raw)
        loss_pp            = ((pred - target_norm) ** 2).mean(dim=-1)
        mw                 = weights[m_mask]
        loss               = (loss_pp[m_mask] * mw).sum() / mw.sum()
        return loss, pred, m_mask


mae   = ViTMAE().to(DEVICE)
dummy = torch.randn(2, IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
loss, pred, mask = mae(dummy)
n_tot = sum(p.numel() for p in mae.parameters() if p.requires_grad)
n_enc = sum(p.numel() for p in mae.encoder.parameters() if p.requires_grad)
n_dec = sum(p.numel() for p in mae.decoder.parameters() if p.requires_grad)
print(f'Forward OK  loss={loss.item():.4f}  pred={pred.shape}')
print(f'Total params : {n_tot:,}')
print(f'  Encoder    : {n_enc:,}  (saved to vit-multitask)')
print(f'  Decoder    : {n_dec:,}  (discarded)')
del dummy, loss, pred, mask


/var/folders/jc/3b0b9y_n1tl_z_b6y_6_k9800000gn/T/ipykernel_28058/422697316.py:22: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=decoder_depth)


Forward OK  loss=1.0338  pred=torch.Size([2, 625, 200])
Total params : 4,264,040
  Encoder    : 3,718,464  (saved to vit-multitask)
  Decoder    : 545,576  (discarded)


## 9 . Optimiser & LR Schedule

In [13]:
def build_mae_opt(model):
    no_decay  = {'bias','LayerNorm.weight','LayerNorm.bias',
                 'norm.weight','norm.bias','pos_embed','cls_token'}
    decay_p   = [p for n,p in model.named_parameters()
                 if p.requires_grad and not any(nd in n for nd in no_decay)]
    nodecay_p = [p for n,p in model.named_parameters()
                 if p.requires_grad and     any(nd in n for nd in no_decay)]
    opt = optim.AdamW([{'params':decay_p,   'weight_decay':WEIGHT_DECAY},
                       {'params':nodecay_p, 'weight_decay':0.0}],
                      lr=LR, betas=(0.9, 0.95))
    def lr_lambda(ep):
        if ep < WARMUP_EPOCHS: return (ep+1)/WARMUP_EPOCHS
        t = (ep-WARMUP_EPOCHS)/max(1, EPOCHS-WARMUP_EPOCHS)
        return 0.5*(1+math.cos(math.pi*t))
    return opt, optim.lr_scheduler.LambdaLR(opt, lr_lambda)


## 10 . Training Loop

In [14]:
def train_epoch_mae(model, loader, opt, scaler):
    model.train()
    total_loss, n = 0.0, 0
    nb = len(loader)
    for i, imgs in enumerate(loader):
        imgs = imgs.to(DEVICE)
        opt.zero_grad()
        with torch.amp.autocast(DEVICE.type, enabled=(DEVICE.type=='cuda')):
            loss, _, _ = model(imgs)
        scaler.scale(loss).backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        bs = imgs.shape[0]; total_loss += loss.item()*bs; n += bs
        bar = 'X'*int(30*(i+1)/nb) + '.'*(30-int(30*(i+1)/nb))
        print(f'\r  [{bar}] {i+1}/{nb}  loss={total_loss/n:.5f}', end='', flush=True)
    print()
    return total_loss / n


## 11 . Run Pretraining

In [15]:
mae_model = ViTMAE().to(DEVICE)
optimizer, scheduler = build_mae_opt(mae_model)
scaler    = torch.amp.GradScaler(enabled=(DEVICE.type=='cuda'))

print(f'ViT-MAE pretraining  |  {EPOCHS} epochs  |  mask {MASK_RATIO:.0%}  |  warmup {WARMUP_EPOCHS}')
print(f'{"Ep":>5} {"Loss":>10} {"LR":>10}  Time')
print('-'*42)

history_loss = []
best_loss, best_ep = float('inf'), 0

for ep in range(1, EPOCHS+1):
    t0   = time.time()
    loss = train_epoch_mae(mae_model, dataloader, optimizer, scaler)
    scheduler.step()
    history_loss.append(loss)
    lr_now = optimizer.param_groups[0]['lr']
    flag   = ' << best' if loss < best_loss else ''
    print(f'{ep:>5}  {loss:>10.6f}  {lr_now:>10.2e}  {time.time()-t0:.1f}s{flag}')
    if loss < best_loss:
        best_loss, best_ep = loss, ep
        torch.save(mae_model.encoder.state_dict(), 'best_vit_mae.pth')
        torch.save({'epoch':ep,'model':mae_model.state_dict(),
                    'opt':optimizer.state_dict(),'loss':best_loss},
                   'vit_mae_checkpoint.pth')

print(f'\nBest loss {best_loss:.6f} at epoch {best_ep}')
print('Encoder saved -> best_vit_mae.pth')


/var/folders/jc/3b0b9y_n1tl_z_b6y_6_k9800000gn/T/ipykernel_28058/422697316.py:22: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=decoder_depth)


ViT-MAE pretraining  |  1 epochs  |  mask 60%  |  warmup 0
   Ep       Loss         LR  Time
------------------------------------------


/Users/abhirajraje/Desktop/E2E_Linear_ViT/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  [XX............................] 341/3750  loss=0.90027

KeyboardInterrupt: 

## 12 . Pretraining Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(history_loss)+1), history_loss, linewidth=1.5, label='ViT-MAE')
ax.axvline(best_ep, color='red', linestyle='--', label=f'best ep {best_ep}')
ax.set_xlabel('Epoch'); ax.set_ylabel('MAE Loss (MSE masked patches)')
ax.set_title('ViT MAE Pretraining -- Reconstruction Loss')
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig('vit_mae_loss.png', dpi=150); plt.show()


## 13 . Reconstruction Visualisation
Predictions are in patch-normalised space; denormalise per patch before display.

In [ ]:
@torch.no_grad()
def visualise_reconstruction(model, dataset, n_samples=3):
    model.eval()
    ps   = model.patch_size
    H_p  = W_p = IMAGE_SIZE // ps
    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4*n_samples))
    axes = axes.reshape(n_samples, 3)
    for row in range(n_samples):
        img = dataset[row].unsqueeze(0).to(DEVICE)
        loss, pred, masked = model(img)
        target_raw = model.patchify(img)
        _, p_mean, p_std   = model.patch_normalise(target_raw)
        pred_denorm = pred * p_std + p_mean
        B, N, _ = pred_denorm.shape
        rec = pred_denorm.reshape(B, H_p, W_p, IN_CHANNELS, ps, ps)
        rec = rec.permute(0,3,1,4,2,5).contiguous().reshape(B, IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE)
        masked_img = img.clone()
        mask_2d    = masked[0].reshape(H_p, W_p)
        for hp in range(H_p):
            for wp in range(W_p):
                if mask_2d[hp, wp]:
                    masked_img[0,:,hp*ps:(hp+1)*ps,wp*ps:(wp+1)*ps] = 0.0
        orig  = img[0,0].cpu().numpy()
        msk   = masked_img[0,0].cpu().numpy()
        recon = rec[0,0].cpu().numpy().clip(0, 1)
        vmin, vmax = orig.min(), orig.max()
        axes[row,0].imshow(orig,  cmap='viridis', vmin=vmin, vmax=vmax)
        axes[row,0].set_title(f'Original (sample {row})')
        axes[row,1].imshow(msk,   cmap='viridis', vmin=vmin, vmax=vmax)
        axes[row,1].set_title(f'Masked ({model.mask_ratio:.0%})')
        axes[row,2].imshow(recon, cmap='viridis', vmin=vmin, vmax=vmax)
        axes[row,2].set_title(f'Reconstruction  loss={loss.item():.4f}')
        for ax in axes[row]: ax.axis('off')
    plt.suptitle('ViT MAE -- Reconstruction Quality (channel 0, denormalised)', fontsize=13)
    plt.tight_layout()
    plt.savefig('vit_mae_reconstruction.png', dpi=150); plt.show()


mae_model.load_state_dict(
    torch.load('vit_mae_checkpoint.pth', map_location=DEVICE)['model'])
visualise_reconstruction(mae_model, dataset)


## 14 . Files Produced for Phase 2

| File | Contents | Used by |
|---|---|---|
| `best_vit_mae.pth` | ViT encoder `state_dict` | `vit-multitask.ipynb` |
| `vit_mae_checkpoint.pth` | Full checkpoint | Optional resume |
| `vit_mae_loss.png` | Pretraining loss curve | Comparison with XCiT |
| `vit_mae_reconstruction.png` | Reconstruction quality | Reporting |

**Comparison:** ViT and XCiT have the same parameter count, same training protocol,
same mask ratio, same loss, and same decoder. Any downstream performance difference
in `vit-multitask.ipynb` isolates the effect of the attention mechanism.